In [19]:
%pip install pandas numpy requests jsonschema

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Telia Data Engineer Internship - ETL + LLM Prompt Chain

This notebook implements:
1. API extraction from RestCountries and Open-Meteo
2. Data transformation with pandas (junior-style readable loops)
3. A 4-step LLM prompt chain for complex transformations
4. JSON schema validation between prompt-chain steps
5. Test case with expected intermediate outputs
6. CSV exports for raw and cleaned data
7. SQLite database loading with schema dump

## 1) Imports and setup

In [20]:
from pathlib import Path
import json
import os
from datetime import date, timedelta

import numpy as np
import pandas as pd
import requests

# Optional dependency for JSON schema validation.
# If missing, run this once in a cell: %pip install jsonschema
from jsonschema import validate

BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Working directory:", BASE_DIR)
print("Output directory:", OUTPUT_DIR)

Working directory: c:\Users\jakom\OneDrive - Tartu Ülikool\Töölaud\Ülikool\Tehisintellekt\github\Telia data engineer
Output directory: c:\Users\jakom\OneDrive - Tartu Ülikool\Töölaud\Ülikool\Tehisintellekt\github\Telia data engineer\outputs


## 2) Extract - RestCountries API (Europe capitals)

In [21]:
RESTCOUNTRIES_URL = "https://restcountries.com/v3.1/region/europe"

def fetch_europe_capitals():
    response = requests.get(RESTCOUNTRIES_URL, timeout=60)
    response.raise_for_status()
    payload = response.json()

    rows = []
    for country in payload:
        country_name = country.get("name", {}).get("common")

        capitals = country.get("capital") or []
        capital_name = capitals[0] if len(capitals) > 0 else None

        latlng = country.get("capitalInfo", {}).get("latlng") or []
        capital_lat = latlng[0] if len(latlng) > 0 else None
        capital_lng = latlng[1] if len(latlng) > 1 else None

        row = {
            "country": country_name,
            "capital": capital_name,
            "capital_latitude": capital_lat,
            "capital_longitude": capital_lng,
            "population": country.get("population"),
            "area": country.get("area"),
        }
        rows.append(row)

    capitals_df = pd.DataFrame(rows)
    return capitals_df

countries_raw_df = fetch_europe_capitals()
countries_raw_df.to_csv(OUTPUT_DIR / "raw_countries_europe.csv", index=False, encoding="utf-8-sig")

print("Countries fetched:", len(countries_raw_df))
display(countries_raw_df.head(10))

Countries fetched: 53


,country,capital,capital_latitude,capital_longitude,population,area
0,Isle of Man,Douglas,54.15,-4.48,84530,572.0
1,Svalbard and Jan Mayen,Longyearbyen,78.22,15.63,2530,61399.0
2,Denmark,Copenhagen,55.67,12.58,6011488,43094.0
3,Kosovo,Pristina,42.67,21.17,1585566,10908.0
4,Portugal,Lisbon,38.72,-9.13,10749635,92090.0
5,United Kingdom,London,51.50,-0.08,69281437,244376.0
6,Jersey,Saint Helier,49.18,-2.10,103267,116.0
7,Lithuania,Vilnius,54.68,25.32,2894886,65300.0
8,Liechtenstein,Vaduz,47.13,9.52,40900,160.0
9,Bosnia and Herzegovina,Sarajevo,43.87,18.42,3422000,51209.0


## 3) Extract - Open-Meteo Archive API (last 30 days per capital)

In [22]:
OPEN_METEO_ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"

today = date.today()
start_date = today - timedelta(days=30)
end_date = today - timedelta(days=1)

DAILY_FIELDS = [
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",
    "wind_speed_10m_max",
    "sunshine_duration",
]

def fetch_weather_for_capitals(capitals_df):
    all_weather_rows = []
    skipped_rows = []

    for _, row in capitals_df.iterrows():
        country = row["country"]
        capital = row["capital"]
        lat = row["capital_latitude"]
        lon = row["capital_longitude"]

        if pd.isna(capital) or pd.isna(lat) or pd.isna(lon):
            skipped_rows.append({
                "country": country,
                "capital": capital,
                "reason": "Missing capital or coordinates"
            })
            continue

        params = {
            "latitude": float(lat),
            "longitude": float(lon),
            "start_date": start_date.isoformat(),
            "end_date": end_date.isoformat(),
            "daily": ",".join(DAILY_FIELDS),
            "timezone": "UTC"
        }

        try:
            response = requests.get(OPEN_METEO_ARCHIVE_URL, params=params, timeout=60)
            response.raise_for_status()
            weather_json = response.json()

            daily = weather_json.get("daily", {})
            dates = daily.get("time", [])

            for i in range(len(dates)):
                weather_row = {
                    "country": country,
                    "capital": capital,
                    "date": dates[i],
                    "temperature_2m_max": daily.get("temperature_2m_max", [None] * len(dates))[i],
                    "temperature_2m_min": daily.get("temperature_2m_min", [None] * len(dates))[i],
                    "precipitation_sum": daily.get("precipitation_sum", [None] * len(dates))[i],
                    "wind_speed_10m_max": daily.get("wind_speed_10m_max", [None] * len(dates))[i],
                    "sunshine_duration": daily.get("sunshine_duration", [None] * len(dates))[i],
                }
                all_weather_rows.append(weather_row)

        except Exception as ex:
            skipped_rows.append({
                "country": country,
                "capital": capital,
                "reason": f"API error: {ex}"
            })

    weather_df = pd.DataFrame(all_weather_rows)
    skipped_df = pd.DataFrame(skipped_rows)
    return weather_df, skipped_df

weather_raw_df, weather_skipped_df = fetch_weather_for_capitals(countries_raw_df)

weather_raw_df.to_csv(OUTPUT_DIR / "raw_weather_daily.csv", index=False, encoding="utf-8-sig")
weather_skipped_df.to_csv(OUTPUT_DIR / "raw_weather_skipped_capitals.csv", index=False, encoding="utf-8-sig")

print("Weather rows fetched:", len(weather_raw_df))
print("Skipped capitals:", len(weather_skipped_df))
display(weather_raw_df.head(10))

Weather rows fetched: 1590
Skipped capitals: 0


,country,capital,date,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,sunshine_duration
0,Isle of Man,Douglas,2026-03-01,9.6,6.4,4.0,44.5,1733.35
1,Isle of Man,Douglas,2026-03-02,9.6,5.8,10.9,44.2,24666.04
2,Isle of Man,Douglas,2026-03-03,8.8,1.0,0.0,14.0,38702.13
3,Isle of Man,Douglas,2026-03-04,10.5,6.3,0.0,18.1,38533.25
4,Isle of Man,Douglas,2026-03-05,10.4,4.3,4.4,36.8,13842.21
5,Isle of Man,Douglas,2026-03-06,7.8,1.6,0.0,22.6,36990.56
6,Isle of Man,Douglas,2026-03-07,7.2,1.2,0.0,18.2,39600.00
7,Isle of Man,Douglas,2026-03-08,8.6,2.3,0.0,17.4,27767.62
8,Isle of Man,Douglas,2026-03-09,9.9,7.2,3.9,21.5,20871.34
9,Isle of Man,Douglas,2026-03-10,9.0,6.3,5.0,42.1,0.00


## 4) Transform - clean and prepare weather data

In [23]:
weather_clean_df = weather_raw_df.copy()

# Normalize data types with explicit loops for readability.
numeric_cols = [
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",
    "wind_speed_10m_max",
    "sunshine_duration",
]

for col in numeric_cols:
    weather_clean_df[col] = pd.to_numeric(weather_clean_df[col], errors="coerce")

weather_clean_df["date"] = pd.to_datetime(weather_clean_df["date"], errors="coerce")

# Convert sunshine from seconds to hours.
weather_clean_df["sunshine_duration_hours"] = weather_clean_df["sunshine_duration"] / 3600.0

# Remove duplicates by country, capital, date (keep first).
before_dedup = len(weather_clean_df)
weather_clean_df = weather_clean_df.drop_duplicates(subset=["country", "capital", "date"], keep="first")
after_dedup = len(weather_clean_df)

# Final sort for stable outputs.
weather_clean_df = weather_clean_df.sort_values(["country", "capital", "date"]).reset_index(drop=True)

weather_clean_df.to_csv(OUTPUT_DIR / "clean_weather_daily.csv", index=False, encoding="utf-8-sig")

print("Rows before dedup:", before_dedup)
print("Rows after dedup:", after_dedup)
display(weather_clean_df.head(10))

Rows before dedup: 1590
Rows after dedup: 1590


,country,capital,date,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,sunshine_duration,sunshine_duration_hours
0,Albania,Tirana,2026-03-01,15.9,5.8,0.0,10.5,37618.89,10.449692
1,Albania,Tirana,2026-03-02,15.8,5.8,0.0,12.5,35465.80,9.851611
2,Albania,Tirana,2026-03-03,16.2,6.6,0.0,11.3,37575.68,10.437689
3,Albania,Tirana,2026-03-04,17.0,7.2,0.0,8.2,38735.29,10.759803
4,Albania,Tirana,2026-03-05,15.9,5.4,0.2,13.0,30339.64,8.427678
5,Albania,Tirana,2026-03-06,17.9,3.9,0.0,9.8,39179.25,10.883125
6,Albania,Tirana,2026-03-07,18.7,7.2,0.0,11.5,38964.27,10.823408
7,Albania,Tirana,2026-03-08,17.0,9.2,1.4,10.6,27223.64,7.562122
8,Albania,Tirana,2026-03-09,15.1,6.2,6.7,7.6,16423.96,4.562211
9,Albania,Tirana,2026-03-10,17.6,2.9,0.0,12.1,40302.47,11.195131


## 5) Views - analytical outputs

In [24]:
# View 1: capitals ranked by average max temperature.
view_avg_temp = (
    weather_clean_df.groupby(["country", "capital"], as_index=False)["temperature_2m_max"]
    .mean()
    .rename(columns={"temperature_2m_max": "avg_temperature_2m_max"})
    .sort_values("avg_temperature_2m_max", ascending=False)
)

# View 2: countries with most rainfall over period.
view_rainfall = (
    weather_clean_df.groupby(["country", "capital"], as_index=False)["precipitation_sum"]
    .sum()
    .rename(columns={"precipitation_sum": "total_precipitation_sum"})
    .sort_values("total_precipitation_sum", ascending=False)
)

# View 3: full 30-day summary per country/capital.
view_30d_summary = weather_clean_df.groupby(["country", "capital"], as_index=False).agg(
    days_count=("date", "count"),
    avg_temp_max=("temperature_2m_max", "mean"),
    avg_temp_min=("temperature_2m_min", "mean"),
    total_rainfall=("precipitation_sum", "sum"),
    max_wind=("wind_speed_10m_max", "max"),
    total_sunshine_hours=("sunshine_duration_hours", "sum"),
)

view_avg_temp.to_csv(OUTPUT_DIR / "view_capitals_ranked_by_avg_temp.csv", index=False, encoding="utf-8-sig")
view_rainfall.to_csv(OUTPUT_DIR / "view_countries_most_rainfall.csv", index=False, encoding="utf-8-sig")
view_30d_summary.to_csv(OUTPUT_DIR / "view_full_30_day_summary.csv", index=False, encoding="utf-8-sig")

print("Saved views to CSV.")
display(view_avg_temp.head(10))

Saved views to CSV.


,country,capital,avg_temperature_2m_max
38,Portugal,Lisbon,18.283333
8,Cyprus,Nicosia,17.840000
33,Montenegro,Podgorica,17.026667
30,Malta,Valletta,16.570000
23,Italy,Rome,16.563333
51,Vatican City,Vatican City,16.563333
32,Monaco,Monaco,16.200000
0,Albania,Tirana,16.096667
45,Spain,Madrid,15.770000
17,Greece,Athens,15.700000


## 6) Prompt chain design (4 step prompts)

In [25]:
transformation_task = "Normalize and deduplicate weather daily records from two APIs (RestCountries + Open-Meteo), with consistent schema and explicit quality checks"

PROMPT_STEP_1 = """
You are a data quality analyst.

Transformation goal:
{transformation_task}

Input data (JSON records):
{raw_data_json}

Task:
Analyze the structure of this data. For each column identify:
1) column_name
2) inferred_type
3) example_values (up to 5)
4) potential_quality_issues

Output format: return ONLY valid JSON with keys: dataset_summary, columns, global_quality_risks.

Error rule:
If ambiguity or parsing error prevents reliable analysis, output: ERROR: [description]
""".strip()

PROMPT_STEP_2 = """
You are a data transformation planner.

Transformation goal:
{transformation_task}

Schema analysis:
{schema_analysis_json}

Task:
Create a numbered transformation plan.
For each step include: step_number, what_to_transform, how_to_transform, why_this_step_is_needed, expected_effect_on_data.

Output format: return ONLY valid JSON with keys: transformation_plan, assumptions, risks.

Error rule:
If goal is ambiguous or conflicts with schema findings, output: ERROR: [description]
""".strip()

PROMPT_STEP_3 = """
You are a data transformation executor.

Transformation goal:
{transformation_task}

Raw data (JSON records):
{raw_data_json}

Approved transformation plan:
{plan_json}

Task:
Execute the transformation plan exactly in order.
For each step show: step_number, operation_applied, short_before_snapshot, short_after_snapshot, row_count_change, column_change_summary.

Output format: return ONLY valid JSON with keys: execution_log, transformed_data_preview, transformed_data_full_reference.

Error rule:
If a step cannot be executed exactly, output: ERROR: [description]
""".strip()

PROMPT_STEP_4 = """
You are a data QA validator.

Transformation goal:
{transformation_task}

Original data (JSON records):
{original_data_json}

Transformed data (JSON records):
{transformed_data_json}

Task:
Validate row count behavior, data loss risk, transformation correctness, schema consistency, and dedup correctness.

Output format: return ONLY valid JSON with keys: validation_status, checks, issues_found, recommended_fixes.

Error rule:
If validation cannot be completed because of malformed input, output: ERROR: [description]
""".strip()

print("Prompt templates are ready.")

Prompt templates are ready.


## 7) JSON schemas for chain validation

In [26]:
STEP1_SCHEMA = {
    "type": "object",
    "required": ["dataset_summary", "columns", "global_quality_risks"],
    "properties": {
        "dataset_summary": {"type": "object"},
        "columns": {"type": "array"},
        "global_quality_risks": {"type": "array"}
    }
}

STEP2_SCHEMA = {
    "type": "object",
    "required": ["transformation_plan", "assumptions", "risks"],
    "properties": {
        "transformation_plan": {"type": "array"},
        "assumptions": {"type": "array"},
        "risks": {"type": "array"}
    }
}

STEP3_SCHEMA = {
    "type": "object",
    "required": ["execution_log", "transformed_data_preview", "transformed_data_full_reference"],
    "properties": {
        "execution_log": {"type": "array"},
        "transformed_data_preview": {"type": "array"},
        "transformed_data_full_reference": {}
    }
}

STEP4_SCHEMA = {
    "type": "object",
    "required": ["validation_status", "checks", "issues_found", "recommended_fixes"],
    "properties": {
        "validation_status": {"type": "string"},
        "checks": {"type": "array"},
        "issues_found": {"type": "array"},
        "recommended_fixes": {"type": "array"}
    }
}

def parse_and_validate_json(text, schema, step_name):
    if isinstance(text, str) and text.startswith("ERROR:"):
        raise ValueError(f"{step_name} returned model error: {text}")

    if isinstance(text, str):
        parsed = json.loads(text)
    else:
        parsed = text

    validate(instance=parsed, schema=schema)
    return parsed

## 8) Orchestration script (with a local mock LLM for deterministic demo)

In [ ]:

def mock_llm_call(step_name, context):
    if step_name == "step1":
        raw_records = context["raw_records"]
        if len(raw_records) == 0:
            return "ERROR: Empty input data"

        df = pd.DataFrame(raw_records)
        columns = []
        for col in df.columns:
            sample_values = []
            for val in df[col].dropna().head(5).tolist():
                sample_values.append(str(val))

            issue_list = []
            if df[col].isna().mean() > 0:
                issue_list.append("Contains null values")
            if df[col].dtype == "object":
                issue_list.append("Potential mixed formats or text normalization needed")

            columns.append({
                "column_name": col,
                "inferred_type": str(df[col].dtype),
                "example_values": sample_values,
                "potential_quality_issues": issue_list
            })

        result = {
            "dataset_summary": {
                "row_count_estimate": int(len(df)),
                "column_count": int(len(df.columns)),
                "notes": ["Schema inferred from sample records"]
            },
            "columns": columns,
            "global_quality_risks": ["Duplicates possible", "Inconsistent casing possible"]
        }
        return json.dumps(result)

    if step_name == "step2":
        plan = {
            "transformation_plan": [
                {
                    "step_number": 1,
                    "what_to_transform": "Normalize text fields",
                    "how_to_transform": "Strip whitespace and title-case country/capital",
                    "why_this_step_is_needed": "Avoid duplicate keys caused by casing and spaces",
                    "expected_effect_on_data": "No row count change"
                },
                {
                    "step_number": 2,
                    "what_to_transform": "Convert numeric columns",
                    "how_to_transform": "Use pandas to_numeric with coercion",
                    "why_this_step_is_needed": "Enable robust aggregations",
                    "expected_effect_on_data": "Potential nulls from invalid values"
                },
                {
                    "step_number": 3,
                    "what_to_transform": "Convert sunshine seconds to hours",
                    "how_to_transform": "Create sunshine_duration_hours = sunshine_duration / 3600",
                    "why_this_step_is_needed": "Human-readable metric",
                    "expected_effect_on_data": "Adds one column"
                },
                {
                    "step_number": 4,
                    "what_to_transform": "Deduplicate by country-capital-date",
                    "how_to_transform": "drop_duplicates keep first",
                    "why_this_step_is_needed": "One record per capital per date",
                    "expected_effect_on_data": "Row count can decrease"
                }
            ],
            "assumptions": ["Country + capital + date is unique business key"],
            "risks": ["Keeping first duplicate can hide conflicting values"]
        }
        return json.dumps(plan)

    if step_name == "step3":
        df = pd.DataFrame(context["raw_records"]).copy()
        execution_log = []

        before_preview = df.head(2).to_dict(orient="records")
        for txt_col in ["country", "capital"]:
            if txt_col in df.columns:
                df[txt_col] = df[txt_col].astype(str).str.strip().str.title()
        execution_log.append({
            "step_number": 1,
            "operation_applied": "Normalize text fields",
            "short_before_snapshot": str(before_preview),
            "short_after_snapshot": str(df.head(2).to_dict(orient="records")),
            "row_count_change": "0",
            "column_change_summary": "No column changes"
        })

        numeric_cols = ["temperature_2m_max", "temperature_2m_min", "precipitation_sum", "wind_speed_10m_max", "sunshine_duration"]
        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")
        execution_log.append({
            "step_number": 2,
            "operation_applied": "Convert numeric columns",
            "short_before_snapshot": "N/A",
            "short_after_snapshot": "Numeric conversion complete",
            "row_count_change": "0",
            "column_change_summary": "No column additions"
        })

        if "sunshine_duration" in df.columns:
            df["sunshine_duration_hours"] = df["sunshine_duration"] / 3600.0
        execution_log.append({
            "step_number": 3,
            "operation_applied": "Create sunshine_duration_hours",
            "short_before_snapshot": "Column absent",
            "short_after_snapshot": "Column created",
            "row_count_change": "0",
            "column_change_summary": "Added sunshine_duration_hours"
        })

        before_rows = len(df)
        if all(k in df.columns for k in ["country", "capital", "date"]):
            df = df.drop_duplicates(subset=["country", "capital", "date"], keep="first")
        after_rows = len(df)
        execution_log.append({
            "step_number": 4,
            "operation_applied": "Deduplicate by country-capital-date",
            "short_before_snapshot": f"Rows: {before_rows}",
            "short_after_snapshot": f"Rows: {after_rows}",
            "row_count_change": str(after_rows - before_rows),
            "column_change_summary": "No column changes"
        })

        output = {
            "execution_log": execution_log,
            "transformed_data_preview": df.head(5).to_dict(orient="records"),
            "transformed_data_full_reference": df.to_json(orient="records")
        }
        return json.dumps(output)

    if step_name == "step4":
        original_df = pd.DataFrame(context["original_records"])
        transformed_df = pd.DataFrame(context["transformed_records"])

        checks = []
        issues_found = []

        original_rows = len(original_df)
        transformed_rows = len(transformed_df)
        row_check_status = "PASS" if transformed_rows <= original_rows else "FAIL"
        checks.append({
            "check_name": "Row count behavior",
            "status": row_check_status,
            "details": f"Original rows={original_rows}, Transformed rows={transformed_rows}"
        })
        if row_check_status == "FAIL":
            issues_found.append("Transformed rows unexpectedly exceed original rows")

        required_cols = ["country", "capital", "date"]
        missing_required = []
        for c in required_cols:
            if c not in transformed_df.columns:
                missing_required.append(c)

        schema_status = "PASS" if len(missing_required) == 0 else "FAIL"
        checks.append({
            "check_name": "Required schema columns",
            "status": schema_status,
            "details": f"Missing: {missing_required}"
        })
        if schema_status == "FAIL":
            issues_found.append("Missing required columns in transformed data")

        dup_count = 0
        if all(c in transformed_df.columns for c in required_cols):
            dup_count = transformed_df.duplicated(subset=required_cols).sum()
        dedup_status = "PASS" if int(dup_count) == 0 else "FAIL"
        checks.append({
            "check_name": "Dedup correctness",
            "status": dedup_status,
            "details": f"Duplicate key rows after transform: {int(dup_count)}"
        })
        if dedup_status == "FAIL":
            issues_found.append("Duplicate rows remain after deduplication")

        validation_status = "PASS" if len(issues_found) == 0 else "FAIL"
        output = {
            "validation_status": validation_status,
            "checks": checks,
            "issues_found": issues_found,
            "recommended_fixes": ["Review failed checks and rerun chain"] if len(issues_found) > 0 else []
        }
        return json.dumps(output)

    return "ERROR: Unknown step name"


def run_prompt_chain(raw_df, llm_call_fn, transformation_task_text):
    raw_records = raw_df.to_dict(orient="records")

    step1_prompt = PROMPT_STEP_1.format(
        transformation_task=transformation_task_text,
        raw_data_json=json.dumps(raw_records[:20], ensure_ascii=False)
    )
    step1_out_text = llm_call_fn("step1", {"prompt": step1_prompt, "raw_records": raw_records})
    step1_out = parse_and_validate_json(step1_out_text, STEP1_SCHEMA, "Step 1")

    step2_prompt = PROMPT_STEP_2.format(
        transformation_task=transformation_task_text,
        schema_analysis_json=json.dumps(step1_out, ensure_ascii=False)
    )
    step2_out_text = llm_call_fn("step2", {"prompt": step2_prompt, "schema_analysis": step1_out})
    step2_out = parse_and_validate_json(step2_out_text, STEP2_SCHEMA, "Step 2")

    step3_prompt = PROMPT_STEP_3.format(
        transformation_task=transformation_task_text,
        raw_data_json=json.dumps(raw_records[:50], ensure_ascii=False),
        plan_json=json.dumps(step2_out, ensure_ascii=False)
    )
    step3_out_text = llm_call_fn("step3", {"prompt": step3_prompt, "raw_records": raw_records, "plan": step2_out})
    step3_out = parse_and_validate_json(step3_out_text, STEP3_SCHEMA, "Step 3")

    transformed_records = json.loads(step3_out["transformed_data_full_reference"])

    step4_prompt = PROMPT_STEP_4.format(
        transformation_task=transformation_task_text,
        original_data_json=json.dumps(raw_records[:50], ensure_ascii=False),
        transformed_data_json=json.dumps(transformed_records[:50], ensure_ascii=False)
    )
    step4_out_text = llm_call_fn("step4", {
        "prompt": step4_prompt,
        "original_records": raw_records,
        "transformed_records": transformed_records
    })
    step4_out = parse_and_validate_json(step4_out_text, STEP4_SCHEMA, "Step 4")

    chain_result = {
        "step1_schema_analysis": step1_out,
        "step2_transformation_plan": step2_out,
        "step3_execution": step3_out,
        "step4_validation": step4_out,
    }
    return chain_result

## 9) Test case + expected intermediate outputs

In [ ]:
test_records = [
    {
        "country": "estonia", "capital": " tallinn ", "date": "2026-03-01",
        "temperature_2m_max": "5.1", "temperature_2m_min": "-1.2",
        "precipitation_sum": "0.5", "wind_speed_10m_max": "22.3", "sunshine_duration": "18000"
    },
    {
        "country": "Estonia", "capital": "Tallinn", "date": "2026-03-01",
        "temperature_2m_max": "5.1", "temperature_2m_min": "-1.2",
        "precipitation_sum": "0.5", "wind_speed_10m_max": "22.3", "sunshine_duration": "18000"
    },
    {
        "country": "Latvia", "capital": "Riga", "date": "2026-03-01",
        "temperature_2m_max": "4.0", "temperature_2m_min": "-2.0",
        "precipitation_sum": "2.1", "wind_speed_10m_max": "18.0", "sunshine_duration": "12000"
    }
]

test_df = pd.DataFrame(test_records)
chain_test_result = run_prompt_chain(test_df, mock_llm_call, transformation_task)

expected_step1_keys = {"dataset_summary", "columns", "global_quality_risks"}
expected_step2_keys = {"transformation_plan", "assumptions", "risks"}

assert expected_step1_keys.issubset(chain_test_result["step1_schema_analysis"].keys())
assert expected_step2_keys.issubset(chain_test_result["step2_transformation_plan"].keys())
assert chain_test_result["step4_validation"]["validation_status"] == "PASS"

transformed_test = pd.DataFrame(json.loads(chain_test_result["step3_execution"]["transformed_data_full_reference"]))
assert len(transformed_test) == 2, "Dedup expected to reduce 3 rows to 2 rows"
assert "sunshine_duration_hours" in transformed_test.columns

print("Test case passed. Prompt chain and validation are working.")
print("Validation output:")
print(json.dumps(chain_test_result["step4_validation"], indent=2))

Test case passed. Prompt chain and validation are working.
Validation output:
{
  "validation_status": "PASS",
  "checks": [
    {
      "check_name": "Row count behavior",
      "status": "PASS",
      "details": "Original rows=3, Transformed rows=2"
    },
    {
      "check_name": "Required schema columns",
      "status": "PASS",
      "details": "Missing: []"
    },
    {
      "check_name": "Dedup correctness",
      "status": "PASS",
      "details": "Duplicate key rows after transform: 0"
    }
  ],
  "issues_found": [],
  "recommended_fixes": []
}


## 10) Run prompt chain on real API data sample and save outputs

In [29]:
def mock_llm_call_safe(step_name, context):
    if step_name != "step3":
        return mock_llm_call(step_name, context)

    df = pd.DataFrame(context["raw_records"]).copy()
    execution_log = []

    before_preview = df.head(2).to_dict(orient="records")
    for txt_col in ["country", "capital"]:
        if txt_col in df.columns:
            df[txt_col] = df[txt_col].astype(str).str.strip().str.title()
    execution_log.append({
        "step_number": 1,
        "operation_applied": "Normalize text fields",
        "short_before_snapshot": str(before_preview),
        "short_after_snapshot": str(df.head(2).to_dict(orient="records")),
        "row_count_change": "0",
        "column_change_summary": "No column changes"
    })

    numeric_cols = ["temperature_2m_max", "temperature_2m_min", "precipitation_sum", "wind_speed_10m_max", "sunshine_duration"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    execution_log.append({
        "step_number": 2,
        "operation_applied": "Convert numeric columns",
        "short_before_snapshot": "N/A",
        "short_after_snapshot": "Numeric conversion complete",
        "row_count_change": "0",
        "column_change_summary": "No column additions"
    })

    if "sunshine_duration" in df.columns:
        df["sunshine_duration_hours"] = df["sunshine_duration"] / 3600.0
    execution_log.append({
        "step_number": 3,
        "operation_applied": "Create sunshine_duration_hours",
        "short_before_snapshot": "Column absent",
        "short_after_snapshot": "Column created",
        "row_count_change": "0",
        "column_change_summary": "Added sunshine_duration_hours"
    })

    before_rows = len(df)
    if all(k in df.columns for k in ["country", "capital", "date"]):
        df = df.drop_duplicates(subset=["country", "capital", "date"], keep="first")
    after_rows = len(df)
    execution_log.append({
        "step_number": 4,
        "operation_applied": "Deduplicate by country-capital-date",
        "short_before_snapshot": f"Rows: {before_rows}",
        "short_after_snapshot": f"Rows: {after_rows}",
        "row_count_change": str(after_rows - before_rows),
        "column_change_summary": "No column changes"
    })

    output = {
        "execution_log": execution_log,
        "transformed_data_preview": df.head(5).to_dict(orient="records"),
        "transformed_data_full_reference": df.to_json(orient="records", date_format="iso")
    }
    return json.dumps(output, default=str)


# Keep sample size manageable for demo prompt context.
real_sample_df = weather_clean_df.head(200).copy()
real_chain_result = run_prompt_chain(real_sample_df, mock_llm_call_safe, transformation_task)

with open(OUTPUT_DIR / "prompt_chain_step1_schema_analysis.json", "w", encoding="utf-8") as f:
    json.dump(real_chain_result["step1_schema_analysis"], f, ensure_ascii=False, indent=2)

with open(OUTPUT_DIR / "prompt_chain_step2_plan.json", "w", encoding="utf-8") as f:
    json.dump(real_chain_result["step2_transformation_plan"], f, ensure_ascii=False, indent=2)

with open(OUTPUT_DIR / "prompt_chain_step3_execution.json", "w", encoding="utf-8") as f:
    json.dump(real_chain_result["step3_execution"], f, ensure_ascii=False, indent=2)

with open(OUTPUT_DIR / "prompt_chain_step4_validation.json", "w", encoding="utf-8") as f:
    json.dump(real_chain_result["step4_validation"], f, ensure_ascii=False, indent=2)

print("Prompt-chain JSON artifacts saved to outputs/.")
print("Final validation status:", real_chain_result["step4_validation"]["validation_status"])

TypeError: Object of type Timestamp is not JSON serializable

## 11) Load into SQLite database and generate schema dump

In [ ]:
import sqlite3

DB_PATH = OUTPUT_DIR / "weather_etl.db"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Create raw_countries table
cursor.execute("""
CREATE TABLE IF NOT EXISTS raw_countries (
    country TEXT NOT NULL,
    capital TEXT,
    capital_latitude REAL,
    capital_longitude REAL,
    population INTEGER,
    area REAL,
    PRIMARY KEY (country)
)
""")

# Create raw_weather table
cursor.execute("""
CREATE TABLE IF NOT EXISTS raw_weather_daily (
    country TEXT NOT NULL,
    capital TEXT NOT NULL,
    date TEXT NOT NULL,
    temperature_2m_max REAL,
    temperature_2m_min REAL,
    precipitation_sum REAL,
    wind_speed_10m_max REAL,
    sunshine_duration REAL,
    PRIMARY KEY (country, capital, date),
    FOREIGN KEY (country) REFERENCES raw_countries(country)
)
""")

# Create clean_weather_daily table
cursor.execute("""
CREATE TABLE IF NOT EXISTS clean_weather_daily (
    country TEXT NOT NULL,
    capital TEXT NOT NULL,
    date TEXT NOT NULL,
    temperature_2m_max REAL,
    temperature_2m_min REAL,
    precipitation_sum REAL,
    wind_speed_10m_max REAL,
    sunshine_duration REAL,
    sunshine_duration_hours REAL,
    PRIMARY KEY (country, capital, date),
    FOREIGN KEY (country) REFERENCES raw_countries(country)
)
""")

# Load CSV data into tables using pandas to_sql
countries_raw_df.to_sql('raw_countries', conn, if_exists='replace', index=False)
weather_raw_df.to_sql('raw_weather_daily', conn, if_exists='replace', index=False)
weather_clean_df.to_sql('clean_weather_daily', conn, if_exists='replace', index=False)

# Create analytical views in the database
cursor.execute("""
CREATE VIEW IF NOT EXISTS v_capitals_ranked_by_avg_temp AS
SELECT country, capital, ROUND(AVG(temperature_2m_max), 2) AS avg_temperature_2m_max
FROM clean_weather_daily
GROUP BY country, capital
ORDER BY avg_temperature_2m_max DESC
""")

cursor.execute("""
CREATE VIEW IF NOT EXISTS v_countries_most_rainfall AS
SELECT country, capital, ROUND(SUM(precipitation_sum), 2) AS total_precipitation_sum
FROM clean_weather_daily
GROUP BY country, capital
ORDER BY total_precipitation_sum DESC
""")

cursor.execute("""
CREATE VIEW IF NOT EXISTS v_30day_summary AS
SELECT 
    country, capital,
    COUNT(date) AS days_count,
    ROUND(AVG(temperature_2m_max), 2) AS avg_temp_max,
    ROUND(AVG(temperature_2m_min), 2) AS avg_temp_min,
    ROUND(SUM(precipitation_sum), 2) AS total_rainfall,
    ROUND(MAX(wind_speed_10m_max), 2) AS max_wind,
    ROUND(SUM(sunshine_duration_hours), 2) AS total_sunshine_hours
FROM clean_weather_daily
GROUP BY country, capital
ORDER BY country
""")

conn.commit()

print(f"SQLite database created at: {DB_PATH}")
print("Tables created: raw_countries, raw_weather_daily, clean_weather_daily")
print("Views created: v_capitals_ranked_by_avg_temp, v_countries_most_rainfall, v_30day_summary")


SQLite database created at: c:\Users\jakom\OneDrive - Tartu Ülikool\Töölaud\Ülikool\Tehisintellekt\github\Telia data engineer\outputs\weather_etl.db
Tables created: raw_countries, raw_weather_daily, clean_weather_daily
Views created: v_capitals_ranked_by_avg_temp, v_countries_most_rainfall, v_30day_summary


## 12) Database schema dump (for deliverables)

In [ ]:
def get_db_schema_dump(db_path):
    """
    Generate a SQL schema dump of the database (suitable for submission as deliverable).
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    schema_dump = ["-- SQLite Database Schema Dump"]
    schema_dump.append(f"-- Generated from: {db_path}")
    schema_dump.append(f"-- Date: {date.today()}")
    schema_dump.append("")
    
    cursor.execute("SELECT sql FROM sqlite_master WHERE type='table' ORDER BY name")
    for row in cursor.fetchall():
        if row[0]:
            schema_dump.append(row[0] + ";")
            schema_dump.append("")
    
    schema_dump.append("-- Views")
    schema_dump.append("")
    cursor.execute("SELECT sql FROM sqlite_master WHERE type='view' ORDER BY name")
    for row in cursor.fetchall():
        if row[0]:
            schema_dump.append(row[0] + ";")
            schema_dump.append("")
    
    cursor.execute("SELECT name, sql FROM sqlite_master WHERE type='index' ORDER BY name")
    for row in cursor.fetchall():
        if row[1]:
            schema_dump.append(row[1] + ";")
            schema_dump.append("")
    
    conn.close()
    return "\n".join(schema_dump)

schema_dump_text = get_db_schema_dump(DB_PATH)
schema_dump_file = OUTPUT_DIR / "database_schema.sql"
with open(schema_dump_file, "w", encoding="utf-8") as f:
    f.write(schema_dump_text)

print(f"Schema dump saved to: {schema_dump_file}")
print("\n--- Preview of Schema ---\n")
print(schema_dump_text)


Schema dump saved to: c:\Users\jakom\OneDrive - Tartu Ülikool\Töölaud\Ülikool\Tehisintellekt\github\Telia data engineer\outputs\database_schema.sql

--- Preview of Schema ---

-- SQLite Database Schema Dump
-- Generated from: c:\Users\jakom\OneDrive - Tartu Ülikool\Töölaud\Ülikool\Tehisintellekt\github\Telia data engineer\outputs\weather_etl.db
-- Date: 2026-03-31

CREATE TABLE "clean_weather_daily" (
"country" TEXT,
  "capital" TEXT,
  "date" TIMESTAMP,
  "temperature_2m_max" REAL,
  "temperature_2m_min" REAL,
  "precipitation_sum" REAL,
  "wind_speed_10m_max" REAL,
  "sunshine_duration" REAL,
  "sunshine_duration_hours" REAL
);

CREATE TABLE "raw_countries" (
"country" TEXT,
  "capital" TEXT,
  "capital_latitude" REAL,
  "capital_longitude" REAL,
  "population" INTEGER,
  "area" REAL
);

CREATE TABLE "raw_weather_daily" (
"country" TEXT,
  "capital" TEXT,
  "date" TEXT,
  "temperature_2m_max" REAL,
  "temperature_2m_min" REAL,
  "precipitation_sum" REAL,
  "wind_speed_10m_max" REAL,
